In [13]:
import ast
import json

import pandas as pd

base_dir = "C:/Users/gabri/OneDrive/Área de Trabalho/joao/TB/cesta-de-precos-pncp"


INPUT = "src/ETL/loaders/catmat.csv"
INPUT_DICIONARIO_OCDS = "tasks/mapeamento-ocds/input/dicionario-medicamentos-ocds.csv"

DATASET_CATMAT = pd.read_csv(f"{base_dir}/{INPUT}")
DATASET_DICIONARIO_OCDS = pd.read_csv(f"{base_dir}/{INPUT_DICIONARIO_OCDS}")

#### Adiciona uma coluna chamada `caracteristicas_ocds`

In [14]:
DATASET_CATMAT['caracteristicas_ocds'] = ''
DATASET_CATMAT.head()

,codigo_classe,nome_classe,codigo_pdm,nome_pdm,codigo_item,nome_item,item_suspenso,item_ativo,item_sustentavel,características,unidades_fornecimento,data_insercao,caracteristicas_ocds
0,6505,DROGAS E MEDICAMENTOS,14597,Petrolato,233632,"Petrolato, Aspecto Físico:Líquido, Tipo:Laxati...",False,True,False,[],"[{""nomeUnidadeMedida"": ""Grama"", ""siglaUnidadeM...",2025-03-20T19:57:26Z,
1,6505,DROGAS E MEDICAMENTOS,8325,Imunoglobulina Humana,260160,"Imunoglobulina Humana, Tipo:Hiper Imuni Para H...",False,True,False,"[{""nomeCaracteristica"": ""Tipo"", ""nomeValorCara...","[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,
2,6505,DROGAS E MEDICAMENTOS,3924,Fenoterol Bromidrato,266532,"Fenoterol Bromidrato, Dosagem:0,2mg / Dose, Ap...",False,True,False,"[{""nomeCaracteristica"": ""Dosagem"", ""nomeValorC...","[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,
3,6505,DROGAS E MEDICAMENTOS,17667,Arteméter,266665,"Arteméter, Dosagem:80 MG/ML, Apresentação:Solu...",False,True,False,[],"[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,
4,6505,DROGAS E MEDICAMENTOS,3946,Budesonida,266699,"Budesonida, Apresentação:Aerossol Bucal, Conce...",False,True,False,[],"[{""capacidadeUnidadeMedida"": 0, ""nomeUnidadeFo...",2025-03-20T19:57:26Z,


#### Funções para montagem da nova coluna

In [15]:
def _valor_ocds_preenchido(valor):
    """Verifica se um valor OCDS vindo do dicionario deve entrar em caracteristicas_ocds."""
    return pd.notna(valor) and str(valor).strip() not in ("", "[]")


def _normaliza_valor_ocds(valor):
    """Remove wrappers de lista serializada e retorna o valor textual usado no JSON final."""
    valor_texto = str(valor).strip()

    # Alguns campos OCDS chegam como lista serializada em texto, por exemplo:
    # '["Líquido"]' ou "['Líquido']". Para o formato final, queremos apenas "Líquido".
    if valor_texto.startswith("[") and valor_texto.endswith("]"):
        try:
            # Primeiro tenta ler como JSON valido: '["Líquido"]'.
            valor_json = json.loads(valor_texto)
        except json.JSONDecodeError:
            try:
                # Se falhar, tenta ler como literal Python seguro: "['Líquido']".
                valor_json = ast.literal_eval(valor_texto)
            except (ValueError, SyntaxError):
                # Se nao for uma lista valida em nenhum formato, preserva o texto original.
                return valor_texto

        if isinstance(valor_json, list):
            # Quando houver mais de um valor, mantem todos em texto unico separado por virgula.
            valores = [str(item).strip() for item in valor_json if str(item).strip()]
            return ", ".join(valores)

    return valor_texto


def _caracteristicas_ocds(row):
    """Monta caracteristicas_ocds no mesmo formato JSON textual da coluna caracteristicas."""
    active_ingredientes = row.get("activeIngredients.name")
    nome_pdm = row.get("nome_pdm_ocds")

    ingredientes = []
    if _valor_ocds_preenchido(nome_pdm):
        ingredientes.append(str(nome_pdm).strip())

    if _valor_ocds_preenchido(active_ingredientes):
        ativos_normalizados = _normaliza_valor_ocds(active_ingredientes)
        for ativo in [item.strip() for item in ativos_normalizados.split(",") if item.strip()]:
            if ativo not in ingredientes:
                ingredientes.append(ativo)

    campos_ocds = [
        ("activeIngredients", ", ".join(ingredientes) if ingredientes else active_ingredientes),
        ("dosageForm", row.get("dosageForm")),
        ("administrationRoute", row.get("administrationRoute")),
        ("strengthValue", row.get("activeIngredients.strengthValue")),
        ("strengthUnit", row.get("activeIngredients.strengthUnit")),
        ("immediateContainer", row.get("immediateContainer")),
    ]

    caracteristicas = [
        {
            "nomeCaracteristica": nome,
            "nomeValorCaracteristica": _normaliza_valor_ocds(valor),
        }
        for nome, valor in campos_ocds
        if _valor_ocds_preenchido(valor)
    ]

    if caracteristicas:
        return json.dumps(caracteristicas, ensure_ascii=False)

    return row.get("caracteristicas_ocds", "[]")

DATASET_CATMAT = DATASET_CATMAT.merge(
    DATASET_DICIONARIO_OCDS,
    left_on="codigo_item",
    right_on="codigo_br",
    how="left",
    suffixes=("", "_ocds"),
)

DATASET_CATMAT["caracteristicas_ocds"] = DATASET_CATMAT.apply(_caracteristicas_ocds, axis=1)

drop_cols = []
for col in DATASET_DICIONARIO_OCDS.columns:
    target_col = col if col in DATASET_CATMAT.columns else f"{col}_ocds"
    if target_col in DATASET_CATMAT.columns:
        drop_cols.append(target_col)


DATASET_CATMAT = DATASET_CATMAT.drop(columns=drop_cols, errors="ignore")

DATASET_CATMAT.head()

,codigo_classe,nome_classe,codigo_pdm,codigo_item,nome_item,item_suspenso,item_ativo,item_sustentavel,características,unidades_fornecimento,data_insercao,caracteristicas_ocds,nome_pdm_ocds
0,6505,DROGAS E MEDICAMENTOS,14597,233632,"Petrolato, Aspecto Físico:Líquido, Tipo:Laxati...",False,True,False,[],"[{""nomeUnidadeMedida"": ""Grama"", ""siglaUnidadeM...",2025-03-20T19:57:26Z,"[{""nomeCaracteristica"": ""activeIngredients"", ""...",Petrolato
1,6505,DROGAS E MEDICAMENTOS,8325,260160,"Imunoglobulina Humana, Tipo:Hiper Imuni Para H...",False,True,False,"[{""nomeCaracteristica"": ""Tipo"", ""nomeValorCara...","[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,"[{""nomeCaracteristica"": ""activeIngredients"", ""...",Imunoglobulina Humana
2,6505,DROGAS E MEDICAMENTOS,3924,266532,"Fenoterol Bromidrato, Dosagem:0,2mg / Dose, Ap...",False,True,False,"[{""nomeCaracteristica"": ""Dosagem"", ""nomeValorC...","[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,"[{""nomeCaracteristica"": ""activeIngredients"", ""...",Fenoterol Bromidrato
3,6505,DROGAS E MEDICAMENTOS,17667,266665,"Arteméter, Dosagem:80 MG/ML, Apresentação:Solu...",False,True,False,[],"[{""nomeUnidadeMedida"": ""Mililitro"", ""siglaUnid...",2025-03-20T19:57:26Z,"[{""nomeCaracteristica"": ""activeIngredients"", ""...",Arteméter
4,6505,DROGAS E MEDICAMENTOS,3946,266699,"Budesonida, Apresentação:Aerossol Bucal, Conce...",False,True,False,[],"[{""capacidadeUnidadeMedida"": 0, ""nomeUnidadeFo...",2025-03-20T19:57:26Z,"[{""nomeCaracteristica"": ""activeIngredients"", ""...",Budesonida
